In [11]:
import os
import time
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import GroupShuffleSplit

In [12]:
# 1. Get the notebook directory
notebook_dir = os.getcwd()

# 2. Go up twice (../../), then into data/processed
# os.path.abspath cleans up the path so it looks pretty in error messages
data_path = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed", "pr_snapshots_clean_v2.csv"))

# 3. Read the file
df = pd.read_csv(data_path)

In [13]:
TARGET_ROWS = 30000
TOLERANCE = 0.15  # skip any single repo that alone would push total too far past target



repo_sizes = df.groupby("repo_key").size().sample(frac=1, random_state=42)
keep_repos, total = [], 0
for repo, size in repo_sizes.items():
    if total >= TARGET_ROWS:
        break
    if total + size > TARGET_ROWS * (1 + TOLERANCE):
        continue
    keep_repos.append(repo)
    total += size

df = df[df.repo_key.isin(keep_repos)].copy()
print(f"{len(keep_repos)} repos, {len(df)} rows")

y = df.pop("merged_before_next")
X = df.select_dtypes(include="number").drop(columns=["pr_id", "number", "timeline_data_available"], errors="ignore")
print(f"shape: {X.shape}, target distribution: {y.value_counts().to_dict()}")

# group split: whole repos held out, never split across train/test
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["repo_key"]))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
print(f"train repos: {df.repo_key.iloc[train_idx].nunique()}, test repos: {df.repo_key.iloc[test_idx].nunique()}")

5 repos, 30867 rows
shape: (30867, 24), target distribution: {0: 19349, 1: 11518}
train repos: 4, test repos: 1


In [14]:

# (Keep all your other scoring metric imports here)

# 1. Compute your training sample weights
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)

# 2. FIX: Impute missing values for BOTH train and test splits
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

models = {
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

results = []

for name, model in models.items():
    start_time = time.time()
    
    # 3. FIX: Fit and predict using the IMPUTED dataframes
    model.fit(X_train_imp, y_train, sample_weight=sw_train)
    
    y_pred = model.predict(X_test_imp)
    y_prob = model.predict_proba(X_test_imp)
    
    roc_auc = roc_auc_score(y_test, y_prob[:, 1]) if len(set(y)) == 2 else roc_auc_score(y_test, y_prob, multi_class='ovr')
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "ROC AUC": roc_auc,
        "F1 Score": f1_score(y_test, y_pred, average='weighted'),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "Time Taken (s)": time.time() - start_time
    })

benchmark_df = pd.DataFrame(results).sort_values("Balanced Accuracy", ascending=False).set_index("Model")
display(benchmark_df.round(4))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken (s)
Model,,,,,,,
AdaBoost,0.6203,0.6602,0.6811,0.6257,0.7025,0.6203,2.0077
LightGBM,0.6256,0.6570,0.6929,0.6325,0.6943,0.6256,2.1065
Random Forest,0.6252,0.6392,0.6843,0.6336,0.6714,0.6252,6.3345
XGBoost,0.6114,0.6378,0.6876,0.6191,0.6744,0.6114,0.4172
Extra Trees,0.6276,0.5953,0.6719,0.6289,0.6303,0.6276,4.6658


In [15]:

from sklearn.model_selection import RandomizedSearchCV
# 1. Impute Data
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

# 2. Compute Sample Weights (For everyone except Extra Trees)
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)

# 3. Model & Parameter Configurations
models_config = {
    "AdaBoost": (
        AdaBoostClassifier(random_state=42), 
        {"sample_weight": sw_train}, 
        {"n_estimators": [50, 100, 200], "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]}
    ),
    "LightGBM": (
        LGBMClassifier(random_state=42, verbose=-1), 
        {"sample_weight": sw_train}, 
        {"max_depth": [3, 5, 7, -1], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 1.0]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42), 
        {"sample_weight": sw_train}, 
        {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15, None], "min_samples_split": [2, 5, 10]}
    ),
    "XGBoost": (
        XGBClassifier(eval_metric='logloss', random_state=42), 
        {"sample_weight": sw_train}, 
        {"max_depth": [3, 5, 7], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 1.0]}
    ),
    "Extra Trees": (
        ExtraTreesClassifier(random_state=42), 
        {},  # Vanilla run: No weights passed
        {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15, None], "min_samples_split": [2, 5, 10]}
    )
}

# 4. Tuning & Evaluation Loop
results = []

for name, (model, fit_params, params) in models_config.items():
    start_time = time.time()
    
    # Run RandomCV
    search = RandomizedSearchCV(
        estimator=model, 
        param_distributions=params, 
        n_iter=10, 
        scoring="balanced_accuracy", 
        cv=3, 
        random_state=42, 
        n_jobs=-1
    )
    search.fit(X_train_imp, y_train, **fit_params)
    
    # Grab the best tuned model
    best_model = search.best_estimator_
    
    # Generate Predictions
    y_pred = best_model.predict(X_test_imp)
    y_prob = best_model.predict_proba(X_test_imp)
    roc_auc = roc_auc_score(y_test, y_prob[:, 1]) if len(set(y_test)) == 2 else roc_auc_score(y_test, y_prob, multi_class='ovr')
    
    # Store Metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "ROC AUC": roc_auc,
        "F1 Score": f1_score(y_test, y_pred, average='weighted'),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "Time Taken (s)": time.time() - start_time,
        "Best Params": str(search.best_params_)
    })

# 5. Display Final Benchmark Table
benchmark_df = pd.DataFrame(results).sort_values("Balanced Accuracy", ascending=False).set_index("Model")
display(benchmark_df.round(4))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken (s),Best Params
Model,,,,,,,,
AdaBoost,0.6285,0.6499,0.6690,0.6365,0.6833,0.6285,30.5242,"{'n_estimators': 50, 'learning_rate': 0.5}"
LightGBM,0.6281,0.6494,0.6755,0.6360,0.6829,0.6281,11.5740,"{'subsample': 0.8, 'n_estimators': 200, 'max_d..."
XGBoost,0.6221,0.6455,0.6846,0.6300,0.6802,0.6221,8.2245,"{'subsample': 0.8, 'n_estimators': 100, 'max_d..."
Random Forest,0.6034,0.6327,0.6641,0.6108,0.6710,0.6034,38.9306,"{'n_estimators': 300, 'min_samples_split': 10,..."
Extra Trees,0.6408,0.6100,0.6881,0.6421,0.6436,0.6408,26.1215,"{'n_estimators': 300, 'min_samples_split': 10,..."


In [19]:

from sklearn.model_selection import RandomizedSearchCV, GroupKFold

from sklearn.utils.class_weight import compute_sample_weight

# 1. Impute Data (Only for Scikit-Learn Models)
imputer = SimpleImputer(strategy="constant", fill_value=-1, add_indicator=True)

# Using get_feature_names_out() is required because add_indicator creates new columns
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=imputer.get_feature_names_out(), index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=imputer.get_feature_names_out(), index=X_test.index)

# 2. Compute Sample Weights
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)

# 3. Model Configs: (Model, Fit Params, CV Params, Needs Imputation Flag)
models_config = {
    "AdaBoost": (
        AdaBoostClassifier(random_state=42), 
        {"sample_weight": sw_train}, 
        {"n_estimators": [50, 100, 200], "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]},
        True
    ),
    "LightGBM": (
        LGBMClassifier(random_state=42, verbose=-1), 
        {"sample_weight": sw_train}, 
        {"max_depth": [3, 5, 7, -1], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 1.0]},
        False # Uses raw NaNs in X_train
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42), 
        {"sample_weight": sw_train}, 
        {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15, None], "min_samples_split": [2, 5, 10]},
        True
    ),
    "XGBoost": (
        XGBClassifier(eval_metric='logloss', random_state=42), 
        {"sample_weight": sw_train}, 
        {"max_depth": [3, 5, 7], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 1.0]},
        False # Uses raw NaNs in X_train
    ),
    "Extra Trees": (
        ExtraTreesClassifier(random_state=42), 
        {},  
        {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15, None], "min_samples_split": [2, 5, 10]},
        True
    )
}

# 4. Tuning & Evaluation Loop
results = []

# Define GroupKFold to prevent PR snapshot leakage
gkf = GroupKFold(n_splits=3)

for name, (model, fit_params, params, needs_imputation) in models_config.items():
    start_time = time.time()
    
    # Route to the correct dataset based on model type
    X_tr = X_train_imp if needs_imputation else X_train
    X_te = X_test_imp if needs_imputation else X_test
    
    # Run RandomCV
    search = RandomizedSearchCV(
        estimator=model, 
        param_distributions=params, 
        n_iter=10, 
        scoring="balanced_accuracy", 
        cv=gkf, 
        random_state=42, 
        n_jobs=-1
    )
    
    train_pr_ids = df["pr_id"].iloc[train_idx]
    search.fit(X_tr, y_train, groups=train_pr_ids, **fit_params) 
    
    best_model = search.best_estimator_
    
    # Generate Predictions
    y_pred = best_model.predict(X_te)
    y_prob = best_model.predict_proba(X_te)
    roc_auc = roc_auc_score(y_test, y_prob[:, 1]) if len(set(y_test)) == 2 else roc_auc_score(y_test, y_prob, multi_class='ovr')
    
    # Store Metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "ROC AUC": roc_auc,
        "F1 Score": f1_score(y_test, y_pred, average='weighted'),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "Time Taken (s)": time.time() - start_time,
        "Best Params": str(search.best_params_)
    })

# 5. Display Final Benchmark Table
benchmark_df = pd.DataFrame(results).sort_values("Balanced Accuracy", ascending=False).set_index("Model")
display(benchmark_df.round(4))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken (s),Best Params
Model,,,,,,,,
Random Forest,0.6244,0.6583,0.6944,0.6309,0.6969,0.6244,46.9255,"{'n_estimators': 300, 'min_samples_split': 10,..."
LightGBM,0.6274,0.6576,0.6933,0.6345,0.6942,0.6274,10.2686,"{'subsample': 0.8, 'n_estimators': 200, 'max_d..."
AdaBoost,0.6184,0.6483,0.6895,0.6256,0.6855,0.6184,30.3976,"{'n_estimators': 200, 'learning_rate': 1.0}"
XGBoost,0.6122,0.6411,0.6858,0.6196,0.6785,0.6122,6.5368,"{'subsample': 0.8, 'n_estimators': 300, 'max_d..."
Extra Trees,0.6383,0.6237,0.6860,0.6439,0.6543,0.6383,24.2673,"{'n_estimators': 200, 'min_samples_split': 5, ..."


In [21]:

# 1. Load Data
notebook_dir = os.getcwd()
data_path = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed", "pr_snapshots_clean_v2.csv"))
df = pd.read_csv(data_path)

# Create a working copy to preserve the original 'df'
working_df = df.copy()

# 2. Prepare Features and Target
y = working_df.pop("merged_before_next")
X = working_df.select_dtypes(include="number").drop(columns=["pr_id", "number", "timeline_data_available"], errors="ignore")
print(f"shape: {X.shape}, target distribution: {y.value_counts().to_dict()}")

# 3. Group Split: Hold out exactly 2 whole repos for testing
splitter = GroupShuffleSplit(n_splits=1, test_size=2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=working_df["repo_key"]))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print(f"train repos: {working_df.repo_key.iloc[train_idx].nunique()}, test repos: {working_df.repo_key.iloc[test_idx].nunique()}")

# 4. Extract PR IDs specifically for the inner GroupKFold tuning
train_pr_ids = working_df["pr_id"].iloc[train_idx]

# 5. Impute Data (with -1 constant and indicator flag)
imputer = SimpleImputer(strategy="constant", fill_value=-1, add_indicator=True)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=imputer.get_feature_names_out(), index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=imputer.get_feature_names_out(), index=X_test.index)

# 6. Compute Sample Weights for balanced training
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)

# 7. Configure Random Forest and RandomizedSearchCV
model = RandomForestClassifier(random_state=42)
params = {
    "n_estimators": [100, 200, 300], 
    "max_depth": [5, 10, 15, None], 
    "min_samples_split": [2, 5, 10]
}

gkf = GroupKFold(n_splits=3)

search = RandomizedSearchCV(
    estimator=model, 
    param_distributions=params, 
    n_iter=10, 
    scoring="balanced_accuracy", 
    cv=gkf, 
    random_state=42, 
    n_jobs=-1
)

# 8. Train and Tune
print("Training Random Forest...")
start_time = time.time()

# Pass groups to prevent PR leakage, and sample_weight to handle imbalance
search.fit(X_train_imp, y_train, groups=train_pr_ids, sample_weight=sw_train)
best_model = search.best_estimator_

# 9. Evaluate
y_pred = best_model.predict(X_test_imp)
y_prob = best_model.predict_proba(X_test_imp)
roc_auc = roc_auc_score(y_test, y_prob[:, 1]) if len(set(y_test)) == 2 else roc_auc_score(y_test, y_prob, multi_class='ovr')

# 10. Display Results
results = pd.DataFrame([{
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "ROC AUC": roc_auc,
    "F1 Score": f1_score(y_test, y_pred, average='weighted'),
    "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
    "Time Taken (s)": time.time() - start_time,
    "Best Params": str(search.best_params_)
}])

display(results.round(4))

shape: (364705, 24), target distribution: {0: 259682, 1: 105023}
train repos: 39, test repos: 2
Training Random Forest...


,Model,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken (s),Best Params
0,Random Forest,0.6915,0.6885,0.7561,0.6923,0.6938,0.6915,963.1213,"{'n_estimators': 300, 'min_samples_split': 10,..."
